# 06 Sea-feature identification

At P-band the sea is seen through surface ripples about a metre long. Anything that changes them changes how bright the sea looks: rain and surface films smooth them and make the sea darker, stronger wind or a current front roughens them. A dark or bright patch on its own doesn't say what caused it, so this notebook (report Section 4.5):

1. finds patches that are clearly brighter or darker than their surroundings and bigger than speckle
2. compares each one with a ring of sea just outside it, in every polarimetric quantity
3. brings in IMERG rain, ERA5 wind, OSTIA SST and the CMEMS salinity analysis at the time of the pass
4. decides what it most likely is, using the rules in Table 3 of the report

Then a second, simpler test that found the El Niño front: chop the frame into a 9 x 9 grid of blocks and look for the block that doesn't scatter like the rest.

It reads the `.npz` from notebook 02 and, for one check, the original product.

In [ ]:
# ---- the only cell that needs editing ----
from pathlib import Path

NPZ = Path(r"C:\Users\Orlaith.Doyle\biomass_processed\T033_F156_20260630T1812.npz")
PRODUCT = Path(r"C:\Users\Orlaith.Doyle\BIO_S1_SCS__1S_20260630T181207_20260630T181228_C_G01_M01_C01_T033_F156_01_DAB2C1")
OUT = Path(r"C:\Users\Orlaith.Doyle\biomass_features")

# ancillary, set any of them to None to skip
ERA5 = Path(r"C:\Users\Orlaith.Doyle\refs\era5_wind_20260630.nc")
IMERG_DIR = Path(r"C:\Users\Orlaith.Doyle\refs\imerg")
OSTIA = Path(r"C:\Users\Orlaith.Doyle\refs\METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2_20260630.nc")
GLORYS_SSS = Path(r"C:\Users\Orlaith.Doyle\refs\cmems_mod_glo_phy-so_anfc_0.083deg_P1D-m_20260630.nc")

# detection
BG_KM = 15.0          # size of the smoothed background
DARK_DB, BRIGHT_DB = 2.0, 2.0   # how far below / above the background counts
MIN_KM2 = 4.0         # anything smaller is left to speckle
RING_KM = 3.0         # width of the ring of sea used for comparison
NESZ_DB = -27.0       # Biomass noise floor

# classification thresholds (Table 3 in the report, made explicit)
SEA_LIKE_DH = 0.10    # entropy rise allowed for "still scatters like sea"
SEA_LIKE_DCOH = 0.03  # coherence drop allowed
POINT_RATIO = 3.0     # 90th / 50th percentile amplitude above this = a few very bright targets
RAIN_MMH = 1.0        # IMERG rain inside the patch to call it rain ...
RAIN_FACTOR = 2.0     # ... and this many times the rain outside
LOW_WIND = 3.0        # m/s
THIN = 4.0            # length / width above this = streak
SST_FRONT = 0.2       # K difference in / out to back a front
SSS_FRONT = 0.2       # psu difference in / out to back a front

GRID_N = 9            # the 9 x 9 block test

In [ ]:
import sys
sys.path.append(str(Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
import bm_tools as bm

OUT.mkdir(parents=True, exist_ok=True)
d = dict(np.load(NPZ, allow_pickle=True))
TAG = str(d["tag"])
t_pass = np.datetime64(str(d["t0"]))
LAT, LON = d["lat"], d["lon"]
cell_m = np.sqrt(float(d["naz"]) * float(d["d_az"]) * float(d["nrg"]) * float(d["d_rg"]))
water = np.isfinite(d["s0vv"])
vvdb = 10 * np.log10(d["s0vv"])
print(TAG, "| cells", vvdb.shape, "| cell about %.0f m" % cell_m)

def save(fig, what):
    fig.savefig(OUT / f"{TAG}_{what}.png", dpi=200, bbox_inches="tight")
    print("saved", f"{TAG}_{what}.png")

## 1. Finding candidate features

The VV image (dB) gets a smoothed background, a median over about `BG_KM`, so slow changes across the swath (incidence angle, large scale wind) aren't counted as features. Anything clearly darker or brighter than that background, cleaned up a little with an opening and closing so single noisy cells don't count, and bigger than `MIN_KM2`, is a candidate.

In [ ]:
nb = max(3, int(BG_KM * 1000 / cell_m) | 1)
fill = np.where(water, vvdb, np.nanmedian(vvdb))
bg = ndimage.median_filter(fill, size=nb)
anom = np.where(water, vvdb - bg, 0)

feats = []
for kind, m in [("dark", anom < -DARK_DB), ("bright", anom > BRIGHT_DB)]:
    m = ndimage.binary_closing(ndimage.binary_opening(m, iterations=1), iterations=2) & water
    lab, n = ndimage.label(m)
    for i in range(1, n + 1):
        f = lab == i
        if f.sum() * cell_m**2 / 1e6 >= MIN_KM2:
            feats.append({"kind": kind, "mask": f})
feats.sort(key=lambda f: -f["mask"].sum())
print(len(feats), "candidate features")

## 2. Describing each feature against its ring

For every feature a ring of sea `RING_KM` wide is taken just outside it (other features and land left out). Inside and ring are compared on:

- brightness in HH, VV and cross-pol
- HH-VV coherence and the spread of the CPD (both high and narrow for normal sea)
- entropy and mean alpha (low and low for normal sea)
- the 90th to 50th percentile ratio of single-look VV amplitude, high when a few very bright pixels (ships, rocks) dominate. This one goes back to the full resolution product
- shape: area and how long and thin it is

Averages of brightness and coherence are taken on the linear second order products, not on dB or on per-cell coherence, and the CPD spread is circular.

In [ ]:
ring_px = max(1, int(RING_KM * 1000 / cell_m))
all_feat = np.zeros(water.shape, bool)
for f in feats:
    all_feat |= f["mask"]

slc = None
if PRODUCT and Path(PRODUCT).exists():
    slc, _ = bm.read_slc(PRODUCT, pols=("VV",))
naz, nrg = int(d["naz"]), int(d["nrg"])


def stats(m):
    hh, vv, x = np.nansum(d["hh"][m]), np.nansum(d["vv"][m]), np.nansum(d["x"][m])
    sinI = np.nanmean(np.sin(np.radians(d["inc"][m])))
    cpd = d["cpd"][m]; cpd = cpd[np.isfinite(cpd)]
    return {"HH_dB": 10 * np.log10(hh / m.sum() * sinI), "VV_dB": 10 * np.log10(vv / m.sum() * sinI),
            "X_dB": 10 * np.log10(x / m.sum() * sinI),
            "coh": np.abs(np.nansum(d["hhvv"][m])) / np.sqrt(hh * vv),
            "cpd_sd": np.degrees(bm.circ_std(cpd)) if cpd.size else np.nan,
            "H": np.nanmedian(d["H"][m]), "alpha": np.nanmedian(d["alpha"][m])}


def amp_ratio(m, cap=200000):
    if slc is None:
        return np.nan
    ii, jj = np.nonzero(m)
    a = np.concatenate([np.abs(slc["VV"][i * naz:(i + 1) * naz, j * nrg:(j + 1) * nrg]).ravel() for i, j in zip(ii, jj)])
    if a.size > cap:
        a = np.random.default_rng(0).choice(a, cap, replace=False)
    p90, p50 = np.percentile(a, [90, 50])
    return p90 / p50


def shape_of(m):
    ii, jj = np.nonzero(m)
    y, x = ii * naz * float(d["d_az"]), jj * nrg * float(d["d_rg"])
    ev = np.sort(np.linalg.eigvalsh(np.cov(np.vstack([y, x])))) if len(ii) > 2 else [1, 1]
    return m.sum() * cell_m**2 / 1e6, np.sqrt(ev[1] / max(ev[0], 1e-9))


for k, f in enumerate(feats):
    grow = ndimage.binary_dilation(f["mask"], iterations=ring_px)
    ring = grow & ~all_feat & water
    f["ring"] = ring
    f["inside"], f["outside"] = stats(f["mask"]), stats(ring)
    f["p90_p50"], f["p90_p50_ring"] = amp_ratio(f["mask"]), amp_ratio(ring)
    f["area_km2"], f["elong"] = shape_of(f["mask"])
    f["lat"], f["lon"] = np.mean(LAT[f["mask"]]), np.mean(LON[f["mask"]])
    print(f"  feature {k + 1}/{len(feats)}: {f['kind']}, {f['area_km2']:.0f} km2")

## 3. Weather and ocean data at the time of the pass

Each feature and its ring get the IMERG rain rate (half hour closest to the pass, time taken from the file name), ERA5 10 m wind (hour closest), OSTIA SST and the CMEMS salinity analysis (daily). IMERG brackets the pass rather than matching it, so the rain timing is approximate, and the daily SST and salinity can't see anything short lived.

In [ ]:
import xarray as xr

anc = {}
if ERA5 and Path(ERA5).exists():
    e = bm.nearest_time(xr.open_dataset(ERA5), t_pass)
    anc["wind"] = np.hypot(bm.interp_ll(e["u10"], LAT, LON), bm.interp_ll(e["v10"], LAT, LON))
if IMERG_DIR and Path(IMERG_DIR).exists():
    try:
        la, lo, pr, t_im = bm.imerg_nearest(IMERG_DIR, t_pass)
        anc["rain"] = bm.sample_grid(la, lo, pr, LAT, LON)
        print("IMERG half hour centred", t_im, "for a pass at", t_pass)
    except FileNotFoundError as err:
        print(err)
if OSTIA and Path(OSTIA).exists():
    o = bm.nearest_time(xr.open_dataset(OSTIA), t_pass)
    anc["sst"] = bm.interp_ll(o["analysed_sst"], LAT, LON) - 273.15
if GLORYS_SSS and Path(GLORYS_SSS).exists():
    g = bm.nearest_time(xr.open_dataset(GLORYS_SSS), t_pass)
    so = g["so"].isel(depth=0) if "depth" in g["so"].dims else g["so"]
    anc["sss"] = bm.interp_ll(so, LAT, LON)
print("ancillary loaded:", list(anc) or "none")

for f in feats:
    for k, v in anc.items():
        f[k + "_in"] = np.nanmean(v[f["mask"]])
        f[k + "_out"] = np.nanmean(v[f["ring"]])

## 4. Deciding what it is

Table 3 from the report as code. In order:

- too close to the noise floor to say anything: **undetermined**
- a few very bright pixels (high 90/50 ratio): **point target** (ship or small island)
- doesn't scatter like sea any more (entropy up, coherence down): a **front** if SST or salinity changes across it, otherwise undetermined
- still scatters like sea and dark: **rain** if IMERG has rain on it, **low wind** if the wind is very light and it's broad, **surface film** if it's a thin streak with normal wind and no rain
- still scatters like sea and bright: **rougher water** (stronger wind, a gust front from a rain cell, current convergence)

Regular stripes aren't picked up here, they go to the wave chain in notebook 05. And this is a classification made by looking at a few scenes, not something that's been tested statistically.

In [ ]:
def classify(f):
    i, o = f["inside"], f["outside"]
    if i["VV_dB"] < NESZ_DB + 3:
        return "undetermined (near noise floor)"
    if np.isfinite(f["p90_p50"]) and f["p90_p50"] > POINT_RATIO:
        return "point target"
    sea_like = (i["H"] - o["H"] < SEA_LIKE_DH) and (o["coh"] - i["coh"] < SEA_LIKE_DCOH)
    if not sea_like:
        dsst = abs(f.get("sst_in", np.nan) - f.get("sst_out", np.nan))
        dsss = abs(f.get("sss_in", np.nan) - f.get("sss_out", np.nan))
        if dsst > SST_FRONT or dsss > SSS_FRONT:
            return "front"
        return "not sea-like, undetermined"
    if f["kind"] == "dark":
        rin, rout = f.get("rain_in", np.nan), f.get("rain_out", np.nan)
        if rin > RAIN_MMH and rin > RAIN_FACTOR * max(rout, 0.05):
            return "rain"
        if f.get("wind_in", np.nan) < LOW_WIND and f["elong"] < THIN:
            return "low wind"
        if f["elong"] >= THIN and not rin > RAIN_MMH:
            return "surface film"
        return "dark, undetermined"
    return "rougher water (wind / outflow / convergence)"


rows = []
for k, f in enumerate(feats):
    f["class"] = classify(f)
    r = {"id": k + 1, "kind": f["kind"], "class": f["class"], "lat": f["lat"], "lon": f["lon"],
         "area_km2": f["area_km2"], "length_width": f["elong"], "p90_p50": f["p90_p50"], "p90_p50_ring": f["p90_p50_ring"]}
    for q in ("VV_dB", "HH_dB", "X_dB", "coh", "cpd_sd", "H", "alpha"):
        r[q + "_in"], r[q + "_out"] = f["inside"][q], f["outside"][q]
    for q in ("rain", "wind", "sst", "sss"):
        if q + "_in" in f:
            r[q + "_in"], r[q + "_out"] = f[q + "_in"], f[q + "_out"]
    rows.append(r)
tab = pd.DataFrame(rows)
tab.to_csv(OUT / f"{TAG}_features.csv", index=False)
show = [c for c in ["id", "kind", "class", "area_km2", "length_width", "VV_dB_in", "VV_dB_out", "coh_in", "coh_out",
                    "H_in", "H_out", "p90_p50", "rain_in", "rain_out", "wind_in"] if c in tab]
print(tab[show].round(2).to_string(index=False))

In [ ]:
colours = {"rain": "tab:blue", "low wind": "tab:cyan", "surface film": "tab:purple", "front": "tab:red",
           "point target": "yellow", "rougher water (wind / outflow / convergence)": "tab:orange"}

fig, axs = plt.subplots(1, 3, figsize=(17, 7), layout="constrained")
lo_, hi_ = np.nanpercentile(vvdb, [2, 98])
axs[0].pcolormesh(LON, LAT, vvdb, cmap="gray", vmin=lo_, vmax=hi_, shading="auto")
for k, f in enumerate(feats):
    axs[0].contour(LON, LAT, f["mask"].astype(float), levels=[0.5], colors=[colours.get(f["class"], "w")], linewidths=1.2)
    axs[0].text(f["lon"], f["lat"], str(k + 1), color=colours.get(f["class"], "w"), fontsize=9, weight="bold")
axs[0].set_title("VV sigma0 with features")
im = axs[1].pcolormesh(LON, LAT, np.where(water, anom, np.nan), cmap="RdBu_r", vmin=-4, vmax=4, shading="auto")
plt.colorbar(im, ax=axs[1], shrink=0.7, label="dB from background"); axs[1].set_title("VV minus background")
for c, col in colours.items():
    axs[1].plot([], [], color=col, lw=3, label=c)
axs[1].legend(fontsize=7, loc="lower left")
for k, f in enumerate(feats[:6]):
    axs[2].scatter(d["H"][f["mask"]], d["alpha"][f["mask"]], s=3, alpha=0.5, label=f"{k + 1} {f['class'][:14]}")
if feats:
    ring_all = np.zeros_like(water)
    for f in feats:
        ring_all |= f["ring"]
    axs[2].scatter(d["H"][ring_all], d["alpha"][ring_all], s=2, c="k", alpha=0.15, label="rings (sea)")
axs[2].set_xlim(0, 1); axs[2].set_ylim(0, 90); axs[2].set_xlabel("H"); axs[2].set_ylabel("alpha (deg)")
axs[2].legend(fontsize=7, markerscale=4); axs[2].set_title("H-alpha, features vs rings")
for a in axs[:2]:
    a.set_xlabel("lon"); a.set_ylabel("lat")
fig.suptitle(TAG)
save(fig, "features")
plt.show()

if "rain" in anc:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pcolormesh(LON, LAT, vvdb, cmap="gray", vmin=lo_, vmax=hi_, shading="auto")
    cs = ax.contour(LON, LAT, anc["rain"], levels=[0.5, 1, 2, 5, 10], cmap="cool", linewidths=1)
    ax.clabel(cs, fmt="%g mm/h", fontsize=7)
    ax.set_title(f"{TAG}\nVV with IMERG rain contours"); ax.set_xlabel("lon"); ax.set_ylabel("lat")
    save(fig, "rain_overlay")
    plt.show()

## 5. The 9 x 9 block test

The El Niño front wasn't found by the patch detection, it was found by splitting the frame into a 9 x 9 grid and comparing the blocks. One block had coherence 0.84 against 0.92 to 0.94 everywhere else and much more mixed scattering, and it lined up with a small low salinity tongue and an SST front.

Per block this works out the coherence (from the summed cross products), the median entropy and alpha, and flags blocks that sit far from the rest using a robust z-score (median and MAD, so the odd block can't drag the reference around). A block also has to differ by a meaningful amount, the same coherence and entropy steps used for "sea-like" above.

In [ ]:
nr, nc = water.shape
re = np.linspace(0, nr, GRID_N + 1).astype(int)
ce = np.linspace(0, nc, GRID_N + 1).astype(int)
blk = []
for i in range(GRID_N):
    for j in range(GRID_N):
        s = (slice(re[i], re[i + 1]), slice(ce[j], ce[j + 1]))
        w = water[s]
        if w.sum() < 0.3 * w.size:
            continue
        coh = np.abs(np.nansum(d["hhvv"][s])) / np.sqrt(np.nansum(d["hh"][s]) * np.nansum(d["vv"][s]))
        blk.append({"block": f"{chr(65 + i)}{j + 1}", "i": i, "j": j, "n": int(w.sum()), "coh": coh,
                    "H": np.nanmedian(d["H"][s]), "alpha": np.nanmedian(d["alpha"][s])})
blk = pd.DataFrame(blk)

def rz(x):
    mad = 1.4826 * np.median(np.abs(x - np.median(x)))
    return (x - np.median(x)) / (mad if mad > 0 else np.nan)

blk["z_coh"], blk["z_H"] = rz(blk.coh), rz(blk.H)
# needs to be far from the rest AND by an amount that matters, otherwise a very uniform frame
# flags blocks that differ in the third decimal
blk["odd"] = ((blk.z_coh < -3) & (blk.coh < blk.coh.median() - SEA_LIKE_DCOH)) | \
             ((blk.z_H > 3) & (blk.H > blk.H.median() + SEA_LIKE_DH))
blk.to_csv(OUT / f"{TAG}_blocks.csv", index=False)
print("coherence across blocks: %.3f to %.3f" % (blk.coh.min(), blk.coh.max()))
print(blk[blk.odd].round(3).to_string(index=False) if blk.odd.any() else "no block stands out")

fig, axs = plt.subplots(1, 3, figsize=(16, 5), layout="constrained")
for a, q, cm in [(axs[0], "coh", "viridis"), (axs[1], "H", "magma")]:
    M = np.full((GRID_N, GRID_N), np.nan)
    for _, b in blk.iterrows():
        M[b.i, b.j] = b[q]
    im = a.imshow(M, cmap=cm)
    plt.colorbar(im, ax=a, shrink=0.8)
    for _, b in blk[blk.odd].iterrows():
        a.add_patch(plt.Rectangle((b.j - 0.5, b.i - 0.5), 1, 1, fill=False, ec="r", lw=2))
    a.set_xticks(range(GRID_N)); a.set_xticklabels(range(1, GRID_N + 1))
    a.set_yticks(range(GRID_N)); a.set_yticklabels([chr(65 + i) for i in range(GRID_N)])
    a.set_title(f"block {q} (red = stands out), radar geometry")
for _, b in blk.iterrows():
    s = (slice(re[b.i], re[b.i + 1]), slice(ce[b.j], ce[b.j + 1]))
    axs[2].scatter(d["H"][s].ravel(), d["alpha"][s].ravel(), s=1, alpha=0.6 if b.odd else 0.05,
                   c="r" if b.odd else "k", rasterized=True)
axs[2].set_xlim(0, 1); axs[2].set_ylim(0, 90); axs[2].set_xlabel("H"); axs[2].set_ylabel("alpha (deg)")
axs[2].set_title("H-alpha: odd blocks red, rest black")
fig.suptitle(TAG)
save(fig, "blocks")
plt.show()